In [1]:
# ================================================================
# 📰 FAKE NEWS AND TEXT SENTIMENT ANALYZER
# Google Colab - Single Cell Application
# ================================================================

# Install required libraries
!pip -q install transformers torch sentencepiece nltk

import re
import nltk
import torch
from collections import Counter
from transformers import pipeline

nltk.download("punkt", quiet=True)

print("=" * 70)
print("        📰 FAKE NEWS & TEXT SENTIMENT ANALYZER")
print("=" * 70)

# ------------------------------------------------
# 1. LOAD AI MODELS
# ------------------------------------------------

print("\n⏳ Loading AI models...")

# Sentiment model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Emotion model
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=1
)

print("✅ AI models loaded successfully!")


# ------------------------------------------------
# 2. GET USER INPUT
# ------------------------------------------------

print("\n" + "-" * 70)

text = input("""
Paste your news/article/text below:

Example:
Scientists have announced a new technology that can produce clean
energy at a lower cost. Researchers said the technology could help
reduce pollution and improve renewable energy production.

Your text:
""")


# ------------------------------------------------
# 3. CHECK INPUT
# ------------------------------------------------

if len(text.strip()) < 20:
    print("\n❌ Please enter at least 20 characters.")
    raise SystemExit


# ------------------------------------------------
# 4. TEXT CLEANING
# ------------------------------------------------

clean_text = re.sub(r"\s+", " ", text).strip()

lower_text = clean_text.lower()

print("\n⏳ Analyzing text...")


# ------------------------------------------------
# 5. SENTIMENT ANALYSIS
# ------------------------------------------------

sentiment_result = sentiment_model(
    clean_text[:512]
)[0]

sentiment_label = sentiment_result["label"]
sentiment_confidence = sentiment_result["score"] * 100

if sentiment_label == "POSITIVE":
    sentiment = "Positive 😊"
elif sentiment_label == "NEGATIVE":
    sentiment = "Negative 😟"
else:
    sentiment = "Neutral 😐"


# ------------------------------------------------
# 6. EMOTION ANALYSIS
# ------------------------------------------------

emotion_result = emotion_model(
    clean_text[:512]
)[0][0]

emotion_label = emotion_result["label"]
emotion_confidence = emotion_result["score"] * 100

emotion_names = {
    "anger": "Angry 😠",
    "disgust": "Disgusted 🤢",
    "fear": "Fearful 😨",
    "joy": "Happy 😊",
    "neutral": "Neutral 😐",
    "sadness": "Sad 😢",
    "surprise": "Surprised 😮"
}

emotion = emotion_names.get(
    emotion_label.lower(),
    emotion_label.capitalize()
)


# ------------------------------------------------
# 7. FAKE NEWS HEURISTIC ANALYSIS
# ------------------------------------------------
# This is a demonstration classifier.
# It is NOT a fact-checking system.
#
# A real fake-news detector requires a trained
# dataset containing REAL and FAKE news articles.


fake_news_indicators = {

    "Sensational Language": [
        "shocking",
        "unbelievable",
        "you won't believe",
        "secret",
        "miracle",
        "exposed",
        "breaking",
        "urgent",
        "bombshell",
        "incredible"
    ],

    "Unverified Claims": [
        "experts say",
        "people say",
        "sources say",
        "according to sources",
        "everyone knows",
        "they don't want you to know",
        "it is believed"
    ],

    "Conspiracy Language": [
        "cover up",
        "conspiracy",
        "hidden truth",
        "secret government",
        "mainstream media",
        "they are hiding",
        "deep state"
    ],

    "Manipulative Language": [
        "share this",
        "share immediately",
        "must share",
        "before it gets deleted",
        "do not ignore",
        "wake up",
        "spread the word"
    ]
}


detected_indicators = []
indicator_count = 0

for category, keywords in fake_news_indicators.items():

    found_keywords = []

    for keyword in keywords:

        if keyword in lower_text:
            found_keywords.append(keyword)
            indicator_count += 1

    if found_keywords:
        detected_indicators.append(
            (category, found_keywords)
        )


# ------------------------------------------------
# 8. SOURCE / CITATION CHECK
# ------------------------------------------------

source_words = [
    "according to",
    "study",
    "research",
    "researchers",
    "university",
    "government",
    "official",
    "report",
    "journal",
    "scientists",
    "data",
    "survey"
]

source_count = 0

for word in source_words:

    if word in lower_text:
        source_count += 1


# ------------------------------------------------
# 9. CLICKBAIT CHECK
# ------------------------------------------------

clickbait_words = [
    "shocking",
    "breaking",
    "you won't believe",
    "secret",
    "miracle",
    "incredible",
    "amazing",
    "must see",
    "urgent",
    "exposed"
]

clickbait_count = 0

for word in clickbait_words:

    if word in lower_text:
        clickbait_count += 1


# ------------------------------------------------
# 10. EXCESSIVE CAPITALIZATION CHECK
# ------------------------------------------------

words_original = text.split()

capital_words = [
    word for word in words_original
    if len(re.sub(r"[^A-Za-z]", "", word)) >= 4
    and re.sub(r"[^A-Za-z]", "", word).isupper()
]

capital_ratio = 0

if len(words_original) > 0:
    capital_ratio = (
        len(capital_words) / len(words_original)
    ) * 100


# ------------------------------------------------
# 11. EXCLAMATION CHECK
# ------------------------------------------------

exclamation_count = text.count("!")

question_count = text.count("?")


# ------------------------------------------------
# 12. FAKE NEWS RISK SCORE
# ------------------------------------------------

fake_score = 0

# Sensational indicators
fake_score += indicator_count * 8

# Clickbait
fake_score += clickbait_count * 5

# Lack of source information
if source_count == 0:
    fake_score += 15

# Excessive capitalization
if capital_ratio > 10:
    fake_score += 10

# Excessive exclamation
if exclamation_count >= 3:
    fake_score += 10

# Limit score
fake_score = min(fake_score, 100)


# ------------------------------------------------
# 13. CLASSIFICATION
# ------------------------------------------------

if fake_score >= 60:

    fake_prediction = "HIGH RISK OF MISLEADING CONTENT 🚨"
    fake_confidence = fake_score

elif fake_score >= 30:

    fake_prediction = "SUSPICIOUS / NEEDS VERIFICATION ⚠️"
    fake_confidence = fake_score

else:

    fake_prediction = "LOW RISK / APPEARS CREDIBLE ✅"
    fake_confidence = 100 - fake_score


# ------------------------------------------------
# 14. KEYWORD EXTRACTION
# ------------------------------------------------

stop_words = {
    "the", "and", "for", "that", "this", "with",
    "from", "have", "has", "had", "was", "were",
    "are", "is", "will", "would", "could",
    "should", "about", "there", "their", "they",
    "them", "then", "than", "into", "your",
    "you", "our", "out", "not", "but", "can",
    "all", "also", "more", "some", "what",
    "when", "where", "which", "who", "why",
    "how", "its", "it's", "his", "her", "been",
    "being", "very", "just", "only", "may",
    "might", "one", "two", "new"
}

words = re.findall(
    r"\b[a-zA-Z]{3,}\b",
    lower_text
)

filtered_words = [
    word for word in words
    if word not in stop_words
]

word_frequency = Counter(filtered_words)

top_keywords = word_frequency.most_common(10)


# ------------------------------------------------
# 15. TEXT STATISTICS
# ------------------------------------------------

word_count = len(words)

character_count = len(clean_text)

sentence_count = len(
    re.findall(r"[.!?]+", clean_text)
)

if sentence_count == 0:
    sentence_count = 1

average_sentence_length = (
    word_count / sentence_count
)


# ------------------------------------------------
# 16. CREDIBILITY CHECK
# ------------------------------------------------

credibility_points = 0

if source_count > 0:
    credibility_points += 25

if fake_score < 30:
    credibility_points += 25

if clickbait_count == 0:
    credibility_points += 15

if capital_ratio <= 10:
    credibility_points += 15

if exclamation_count < 3:
    credibility_points += 10

if word_count >= 50:
    credibility_points += 10

credibility_points = min(
    credibility_points,
    100
)


# ------------------------------------------------
# 17. FINAL RECOMMENDATION
# ------------------------------------------------

if fake_score >= 60:

    recommendation = """
⚠️ Do not share this content without verification.
Check the information using reliable news organizations,
official government websites, research papers, or trusted sources.
"""

elif fake_score >= 30:

    recommendation = """
⚠️ Verify the claims before sharing.
Look for the original source, publication date,
author information, and supporting evidence.
"""

else:

    recommendation = """
✅ The text does not show many common misleading-content
indicators. However, automated analysis cannot prove that
a news article is factually true.
"""


# ------------------------------------------------
# 18. DISPLAY FINAL REPORT
# ------------------------------------------------

print("\n")
print("=" * 70)
print("                    📊 ANALYSIS REPORT")
print("=" * 70)


print("\n📰 FAKE NEWS ANALYSIS")
print("-" * 70)

print("Prediction       :", fake_prediction)
print("Risk Score       :", f"{fake_score}/100")
print("Confidence       :", f"{fake_confidence:.2f}%")
print("Credibility Score:", f"{credibility_points}/100")


print("\n😊 SENTIMENT ANALYSIS")
print("-" * 70)

print("Sentiment        :", sentiment)
print("Confidence       :", f"{sentiment_confidence:.2f}%")


print("\n🎭 EMOTION ANALYSIS")
print("-" * 70)

print("Emotion           :", emotion)
print("Confidence        :", f"{emotion_confidence:.2f}%")


print("\n🔍 CONTENT INDICATORS")
print("-" * 70)

if detected_indicators:

    for category, keywords in detected_indicators:

        print("\n•", category)

        print("  Detected:",
              ", ".join(keywords))

else:

    print("✅ No major suspicious language indicators detected.")


print("\n📚 SOURCE INDICATORS")
print("-" * 70)

print("Source-related terms found:", source_count)

if source_count > 0:
    print("✅ The text contains source/research-related references.")
else:
    print("⚠️ No clear source or research reference detected.")


print("\n📢 CLICKBAIT ANALYSIS")
print("-" * 70)

print("Clickbait indicators:", clickbait_count)
print("Exclamation marks   :", exclamation_count)
print("Question marks      :", question_count)


print("\n🔠 CAPITALIZATION ANALYSIS")
print("-" * 70)

print("Capitalized words:", len(capital_words))
print("Capitalization ratio:", f"{capital_ratio:.2f}%")


print("\n🔑 IMPORTANT KEYWORDS")
print("-" * 70)

if top_keywords:

    for keyword, count in top_keywords:

        print(
            f"• {keyword:<20} {count} occurrence(s)"
        )

else:

    print("No keywords found.")


print("\n📈 TEXT STATISTICS")
print("-" * 70)

print("Characters              :", character_count)
print("Words                   :", word_count)
print("Sentences               :", sentence_count)
print(
    "Average words/sentence :",
    f"{average_sentence_length:.2f}"
)


print("\n💡 RECOMMENDATION")
print("-" * 70)

print(recommendation)


print("=" * 70)
print("              ✅ ANALYSIS COMPLETED")
print("=" * 70)


# ------------------------------------------------
# IMPORTANT PROJECT NOTE
# ------------------------------------------------

print("""
⚠️ PROJECT NOTE:
This demonstration uses NLP sentiment/emotion models and
rule-based credibility indicators.

It does NOT prove whether a news article is factually true.

For a real Fake News Detection project, train a machine-learning
model using a labeled dataset containing REAL and FAKE news.
""")

        📰 FAKE NEWS & TEXT SENTIMENT ANALYZER

⏳ Loading AI models...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ AI models loaded successfully!

----------------------------------------------------------------------

Paste your news/article/text below:

Example:
Scientists have announced a new technology that can produce clean
energy at a lower cost. Researchers said the technology could help
reduce pollution and improve renewable energy production.

Your text:
BREAKING!!! You won't believe this shocking secret! Scientists say that a miracle drink can make people live for 200 years. Share this immediately before the government deletes it!

⏳ Analyzing text...


                    📊 ANALYSIS REPORT

📰 FAKE NEWS ANALYSIS
----------------------------------------------------------------------
Prediction       : HIGH RISK OF MISLEADING CONTENT 🚨
Risk Score       : 83/100
Confidence       : 83.00%
Credibility Score: 40/100

😊 SENTIMENT ANALYSIS
----------------------------------------------------------------------
Sentiment        : Positive 😊
Confidence       : 51.99%

🎭 EMOTION ANALYSIS
----------